# TP4 : de Cassandra à Spark (films OMDb)

Keyspace `movies_cluster`, table `movies` (cluster de 3 nœuds, RF = 3).

## Partie 1 : connexion Spark / Cassandra

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("TP4-Cassandra-Spark")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
sc = spark.sparkContext
print("Version Spark :", spark.version)
print("Master        :", sc.master)
print("Parallélisme  :", sc.defaultParallelism)

Version Spark : 4.2.0
Master        : local[4]
Parallélisme  : 4


In [2]:
%pip install cassandra-driver pyasyncore

Note: you may need to restart the kernel to use updated packages.


In [3]:
from cassandra.cluster import Cluster

cluster = Cluster(["cass1"])
session = cluster.connect()
print("Connexion Cassandra réussie")

KEYSPACE, TABLE = "movies_cluster", "movies"
session.set_keyspace(KEYSPACE)
print("Keyspace :", KEYSPACE, "| Table :", TABLE)

print("\nStructure de la table :")
for r in session.execute("""
    SELECT column_name, type, kind FROM system_schema.columns
    WHERE keyspace_name = 'movies_cluster' AND table_name = 'movies'"""):
    print(" ", r.column_name, "->", r.type, f"({r.kind})")

print("\nQuelques lignes :")
for r in session.execute("SELECT imdb_id, title, year, imdb_rating FROM movies LIMIT 5"):
    print(" ", r)

Connexion Cassandra réussie
Keyspace : movies_cluster | Table : movies

Structure de la table :
  countries -> list<text> (regular)
  directors -> list<text> (regular)
  genres -> list<text> (regular)
  imdb_id -> text (partition_key)
  imdb_rating -> double (regular)
  imdb_votes -> int (regular)
  runtime -> text (regular)
  title -> text (regular)
  year -> int (regular)

Quelques lignes :
  Row(imdb_id='tt0816711', title='World War Z', year=2013, imdb_rating=7.0)
  Row(imdb_id='tt0106364', title='Batman: Mask of the Phantasm', year=1993, imdb_rating=7.8)
  Row(imdb_id='tt0061811', title='In the Heat of the Night', year=1967, imdb_rating=7.9)
  Row(imdb_id='tt0120915', title='Star Wars: Episode I - The Phantom Menace', year=1999, imdb_rating=6.5)
  Row(imdb_id='tt0138097', title='Shakespeare in Love', year=1999, imdb_rating=7.1)


## Partie 2 : DataFrame Spark

In [4]:
from pyspark.sql.types import *

columns = ["imdb_id", "title", "year", "runtime", "genres", "directors",
           "countries", "imdb_rating", "imdb_votes"]
rows = session.execute("SELECT " + ", ".join(columns) + " FROM movies")
data = [tuple(r) for r in rows]

schema = StructType([
    StructField("imdb_id", StringType()),
    StructField("title", StringType()),
    StructField("year", IntegerType()),
    StructField("runtime", StringType()),
    StructField("genres", ArrayType(StringType())),
    StructField("directors", ArrayType(StringType())),
    StructField("countries", ArrayType(StringType())),
    StructField("imdb_rating", DoubleType()),
    StructField("imdb_votes", IntegerType()),
])

df = spark.createDataFrame(data, schema)
df.show(5, truncate=30)
df.printSchema()
print("Nombre de lignes :", df.count())

+---------+------------------------------+----+-------+------------------------------+------------------------------+------------------------------+-----------+----------+
|  imdb_id|                         title|year|runtime|                        genres|                     directors|                     countries|imdb_rating|imdb_votes|
+---------+------------------------------+----+-------+------------------------------+------------------------------+------------------------------+-----------+----------+
|tt0816711|                   World War Z|2013|116 min|   [Action, Adventure, Horror]|                [Marc Forster]|[United States, United King...|        7.0|    778104|
|tt0106364|  Batman: Mask of the Phantasm|1993| 76 min|[Animation, Action, Adventure]|[Kevin Altieri, Boyd Kirkla...|[United States, Japan, Sout...|        7.8|     65207|
|tt0061811|      In the Heat of the Night|1967|110 min|    [Drama, Mystery, Thriller]|              [Norman Jewison]|               [United 

### Données utilisées

100 films OMDb stockés dans Cassandra (`movies_cluster.movies`, RF = 3).
Colonnes importantes : `year`, `imdb_rating`, `imdb_votes`, `runtime`, `genres`.

**Question métier** : comment évolue la note moyenne des films selon la
décennie, et ces moyennes reposent-elles sur assez de films ?

Chaîne : Cassandra → lignes Python → DataFrame Spark.

## Partie 3 : transformations

In [5]:
from pyspark.sql import functions as F

df_sel = df.select("title", "year", "runtime", "imdb_rating", "imdb_votes")
df_sel.show(5, truncate=30)

print("Films très bien notés (note >= 8) :")
df_sel.filter(F.col("imdb_rating") >= 8).show(5, truncate=30)

print("Films très populaires (plus de 500 000 votes) :")
df_sel.filter(F.col("imdb_votes") > 500_000).show(5, truncate=30)

+------------------------------+----+-------+-----------+----------+
|                         title|year|runtime|imdb_rating|imdb_votes|
+------------------------------+----+-------+-----------+----------+
|                   World War Z|2013|116 min|        7.0|    778104|
|  Batman: Mask of the Phantasm|1993| 76 min|        7.8|     65207|
|      In the Heat of the Night|1967|110 min|        7.9|     92488|
|Star Wars: Episode I - The ...|1999|136 min|        6.5|    940514|
|           Shakespeare in Love|1999|123 min|        7.1|    244571|
+------------------------------+----+-------+-----------+----------+
only showing top 5 rows
Films très bien notés (note >= 8) :
+------------------------------+----+-------+-----------+----------+
|                         title|year|runtime|imdb_rating|imdb_votes|
+------------------------------+----+-------+-----------+----------+
|Dr. Strangelove or: How I L...|1964| 95 min|        8.3|    549187|
|Star Wars: Episode IV - A N...|1977|121 mi

In [6]:
df_calc = (df
    .withColumn("decade", F.floor(F.col("year") / 10) * 10)
    .withColumn("runtime_min", F.regexp_extract("runtime", r"(\d+)", 1).cast("int"))
    .withColumn("nb_genres", F.size("genres")))

df_calc.select("title", "year", "decade", "runtime_min", "nb_genres").show(10, truncate=False)

+-----------------------------------------+----+------+-----------+---------+
|title                                    |year|decade|runtime_min|nb_genres|
+-----------------------------------------+----+------+-----------+---------+
|World War Z                              |2013|2010  |116        |3        |
|Batman: Mask of the Phantasm             |1993|1990  |76         |3        |
|In the Heat of the Night                 |1967|1960  |110        |3        |
|Star Wars: Episode I - The Phantom Menace|1999|1990  |136        |3        |
|Shakespeare in Love                      |1999|1990  |123        |3        |
|War for the Planet of the Apes           |2017|2010  |140        |3        |
|Batman v Superman: Dawn of Justice       |2016|2010  |151        |3        |
|Star Wars: Episode VIII - The Last Jedi  |2017|2010  |152        |3        |
|Saturday Night Fever                     |1977|1970  |118        |2        |
|Run All Night                            |2015|2010  |114      

## Partie 4 : actions et agrégations

In [7]:
df_calc.show(5)                                     # action d'affichage
print("Nombre de films :", df_calc.count())         # action de comptage

df_decade = (df_calc
    .filter(F.col("imdb_rating").isNotNull() & F.col("year").isNotNull())
    .groupBy("decade")
    .agg(F.count("*").alias("nb_films"),
         F.round(F.avg("imdb_rating"), 2).alias("note_moyenne"),
         F.max("imdb_rating").alias("note_max"),
         F.round(F.avg("imdb_votes")).alias("votes_moyens"))
    .orderBy("decade"))
df_decade.show(truncate=False)

+---------+--------------------+----+-------+--------------------+--------------------+--------------------+-----------+----------+------+-----------+---------+
|  imdb_id|               title|year|runtime|              genres|           directors|           countries|imdb_rating|imdb_votes|decade|runtime_min|nb_genres|
+---------+--------------------+----+-------+--------------------+--------------------+--------------------+-----------+----------+------+-----------+---------+
|tt0816711|         World War Z|2013|116 min|[Action, Adventur...|      [Marc Forster]|[United States, U...|        7.0|    778104|  2010|        116|        3|
|tt0106364|Batman: Mask of t...|1993| 76 min|[Animation, Actio...|[Kevin Altieri, B...|[United States, J...|        7.8|     65207|  1990|         76|        3|
|tt0061811|In the Heat of th...|1967|110 min|[Drama, Mystery, ...|    [Norman Jewison]|     [United States]|        7.9|     92488|  1960|        110|        3|
|tt0120915|Star Wars: Episod...|19

## Partie 5 : partitions et Lazy Evaluation

In [8]:
print("Partitions initiales :", df.rdd.getNumPartitions())
print("Lignes par partition :", df.rdd.glom().map(len).collect())

df8 = df.repartition(8)
print("Après repartition(8) :", df8.rdd.getNumPartitions())
print("Lignes par partition :", df8.rdd.glom().map(len).collect())

Partitions initiales : 4
Lignes par partition : [25, 25, 25, 25]
Après repartition(8) : 8
Lignes par partition : [13, 12, 14, 12, 12, 12, 13, 12]


In [9]:
# Transformations seulement : aucun calcul n'est lancé
etape1 = df8.filter(F.col("imdb_rating").isNotNull())
etape2 = etape1.withColumn("decade", F.floor(F.col("year") / 10) * 10)
etape3 = etape2.groupBy("decade").agg(F.round(F.avg("imdb_rating"), 2).alias("note_moyenne"))
print("Transformations déclarées : aucun calcul n'a eu lieu.")

Transformations déclarées : aucun calcul n'a eu lieu.


In [10]:
# L'action déclenche l'exécution
etape3.orderBy("decade").show()

+------+------------+
|decade|note_moyenne|
+------+------------+
|  1930|         8.1|
|  1950|         7.9|
|  1960|        7.56|
|  1970|        7.27|
|  1980|        7.73|
|  1990|         6.5|
|  2000|         7.0|
|  2010|        7.02|
|  2020|        6.82|
+------+------------+



### Pourquoi la Lazy Evaluation ?

Les transformations (`filter`, `select`, `withColumn`, `groupBy`) sont seulement
enregistrées dans un plan logique (DAG). Spark n'exécute rien avant une action
(`show`, `count`, `collect`). Il peut ainsi optimiser tout le plan avec Catalyst
(regrouper filtres et sélections, ne lire que les colonnes utiles) et éviter des
calculs inutiles, ce qui compte beaucoup avec de gros volumes.

## Partie 6 : plan d'exécution et shuffle

In [11]:
df_decade.explain()        # plan physique : repérer la ligne Exchange
df_decade.explain(True)    # plans logique, analysé, optimisé et physique

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [decade#98L ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(decade#98L ASC NULLS FIRST, 4), ENSURE_REQUIREMENTS, [plan_id=353]
      +- HashAggregate(keys=[decade#98L], functions=[count(1), avg(imdb_rating#7), max(imdb_rating#7), avg(imdb_votes#8)])
         +- Exchange hashpartitioning(decade#98L, 4), ENSURE_REQUIREMENTS, [plan_id=350]
            +- HashAggregate(keys=[decade#98L], functions=[partial_count(1), partial_avg(imdb_rating#7), partial_max(imdb_rating#7), partial_avg(imdb_votes#8)])
               +- Project [imdb_rating#7, imdb_votes#8, (FLOOR((cast(year#2 as double) / 10.0)) * 10) AS decade#98L]
                  +- Filter (isnotnull(imdb_rating#7) AND isnotnull(year#2))
                     +- Scan ExistingRDD[imdb_id#0,title#1,year#2,runtime#3,genres#4,directors#5,countries#6,imdb_rating#7,imdb_votes#8]


== Parsed Logical Plan ==
'Sort ['decade ASC NULLS FIRST], true
+- Aggregate [decade#98L], [

## Partie 7 : analyse métier, chaîne complète

In [12]:
resultat = (spark.createDataFrame(data, schema)                     # DataFrame issu de Cassandra
    .select("title", "year", "imdb_rating", "imdb_votes")           # sélection
    .filter(F.col("imdb_rating").isNotNull() & (F.col("imdb_votes") > 10_000))  # filtre
    .withColumn("decade", F.floor(F.col("year") / 10) * 10)         # colonne calculée
    .groupBy("decade")                                              # agrégation
    .agg(F.count("*").alias("nb_films"),
         F.round(F.avg("imdb_rating"), 2).alias("note_moyenne"))
    .orderBy("decade"))

print("Plan construit, rien n'est encore exécuté.")
resultat.show()      # action : seul moment où Spark exécute

Plan construit, rien n'est encore exécuté.
+------+--------+------------+
|decade|nb_films|note_moyenne|
+------+--------+------------+
|  1930|       1|         8.1|
|  1950|       1|         7.9|
|  1960|       5|        7.56|
|  1970|       3|        7.27|
|  1980|       6|        7.73|
|  1990|       8|         6.5|
|  2000|      21|         7.0|
|  2010|      46|        7.02|
|  2020|       9|        6.82|
+------+--------+------------+



### Justification

- **Pourquoi ces données** : les films OMDb ont une année et une note, ce qui
  permet de suivre une évolution dans le temps.
- **Pourquoi ces transformations** : le filtre sur les votes écarte les films
  dont la note est peu fiable, et la décennie regroupe les années.
- **Pourquoi cette agrégation** : `avg` donne la note moyenne, `count` indique si
  la moyenne repose sur assez de films (une moyenne sur 2 films n'est pas fiable).
- **Ce qui déclenche l'exécution** : uniquement `show()` ; tout ce qui précède
  construit le plan.
- **Spark UI** : l'action produit un job, avec un stage supplémentaire à cause du
  shuffle du `groupBy` (voir les captures).